# N02 · PyTorch Profiler：把“训练慢”拆成证据链


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

“GPU 利用率低”“训练吞吐不稳定”“某次改动慢了 20%”都不是可执行诊断。Profiler 的价值是把一步训练拆成可观察事件：数据加载、CPU 调度、H2D copy、forward kernel、backward kernel、optimizer、通信、同步等待。

本节目标：你要能回答 **慢在哪里、证据是什么、下一步只改哪个变量**。


## 学习地图与版本说明（截至 2026-04-30）

本节不是教你“打开一个 profiler 截图”，而是训练一种性能诊断顺序：先定义要解释的现象，例如 tokens/sec 降低、step time 抖动、GPU 空洞；再用轻量 timer 缩小阶段；最后用 PyTorch Profiler 采一小段 trace，证明瓶颈在 CPU 调度、数据拷贝、kernel、通信还是同步等待。

版本上，本教程优先参考 PyTorch stable 的 `torch.profiler` API。新版文档强调使用 `torch.profiler.profile(...)`、`schedule(wait/warmup/active/repeat)`、`on_trace_ready`、`tensorboard_trace_handler` 和 `prof.step()` 来控制采样窗口；同时也提醒 `record_shapes`、`with_stack`、`profile_memory` 会引入额外开销。因此课程实验只采短窗口，不把 profiler 当作长期在线监控工具。

学完本节，你应该能交付一份“证据链”：问题是什么、采样窗口是什么、按 self CUDA time / CPU time / memory 看到什么、为什么这能支持你的判断、下一步只改哪一个变量。企业排障中，能把“感觉慢”变成可复现实验，比记住某个 profiler 参数更重要。


## 1. 一步训练到底包含哪些阶段？

以单卡训练为例，一个 step 通常可以拆成：

1. **取数据**：DataLoader worker 读取、解码、collate。
2. **搬数据**：CPU tensor 到 GPU tensor，可能涉及 pinned memory 和 non_blocking copy。
3. **forward**：模型计算 logits/loss。
4. **backward**：autograd 反传，通常比 forward 更重。
5. **optimizer step**：更新参数，AdamW 比 SGD 更重。
6. **日志与同步**：`loss.item()`、打印、显存统计、profiler 可能触发同步。

分布式训练还要加上：DDP all-reduce、FSDP all-gather/reduce-scatter、pipeline send/recv、checkpoint I/O。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = [
    {"阶段": "DataLoader", "耗时ms": 12, "常见根因": "worker 少、解码慢、网络盘慢"},
    {"阶段": "H2D Copy", "耗时ms": 4, "常见根因": "没用 pinned memory / non_blocking"},
    {"阶段": "Forward", "耗时ms": 28, "常见根因": "kernel 形状差、序列太长"},
    {"阶段": "Backward", "耗时ms": 55, "常见根因": "激活多、重算、通信"},
    {"阶段": "Optimizer", "耗时ms": 10, "常见根因": "AdamW 状态大、CPU offload"},
]
df = pd.DataFrame(rows)
display(df)
df.plot.bar(x="阶段", y="耗时ms", legend=False, title="一个训练 step 的教学拆解")
plt.ylabel("ms")
plt.show()


## 2. Profiler 的正确打开方式

PyTorch 官方 profiler 支持：

- 记录 CPU / CUDA activity。
- 使用 schedule 跳过 warmup，只采样稳定阶段。
- 用 `record_function` 给自定义代码段打标签。
- 输出 Chrome trace / TensorBoard trace。

为什么要有 wait/warmup/active？因为刚开始的几个 step 往往包含 kernel 编译、缓存建立、数据预取等额外开销。直接 profile 第 1 step 容易误判。


In [ ]:
# 这个 cell 在 CPU 上也能跑。若有 CUDA，会自动加入 CUDA activity。
try:
    import torch
    from torch.profiler import profile, ProfilerActivity, record_function

    device = "cuda" if torch.cuda.is_available() else "cpu"
    activities = [ProfilerActivity.CPU]
    if device == "cuda":
        activities.append(ProfilerActivity.CUDA)

    x = torch.randn(1024, 1024, device=device)
    with profile(activities=activities, record_shapes=True) as prof:
        with record_function("toy_matmul_relu"):
            y = (x @ x).relu()
            if device == "cuda":
                torch.cuda.synchronize()

    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=8))
except Exception as exc:
    print("Profiler smoke 跳过：", exc)


## 3. 最常见的误判：CUDA 是异步的

CUDA kernel launch 通常是异步的：Python 代码把任务提交到 GPU 后，CPU 可能继续往下走。因此你用 `time.perf_counter()` 包住一段 GPU 代码，可能测到的是“提交任务的时间”，不是“GPU 完成计算的时间”。

工程上：

- 粗略测 GPU 段耗时时，在计时前后 `torch.cuda.synchronize()`。
- 不要在每个 step 频繁 `loss.item()`，它会把 GPU 结果同步回 CPU。
- profiler trace 中要区分 CPU launch 时间和 CUDA kernel 执行时间。


## 4. 如何读 trace：先看大块，再看细节

建议顺序：

1. **总览**：哪个阶段占比最大？DataLoader？forward？backward？optimizer？通信？
2. **空白区**：GPU timeline 是否有大段 idle？如果有，CPU/DataLoader/同步可能是瓶颈。
3. **形状**：record_shapes 后查看异常大的 tensor shape。
4. **调用栈**：从耗时 op 回到模型层或数据处理函数。
5. **对比**：同一 workload 下比较两个 trace，不要拿不同 batch/seq 的 trace 乱比。


In [ ]:
# 一个简单的“瓶颈分类器”：根据分段耗时给出下一步建议。
def diagnose_step(row):
    total = sum(row.values())
    ratios = {k: v / total for k, v in row.items()}
    worst = max(ratios, key=ratios.get)
    suggestions = {
        "dataloader": "先增加 num_workers / pin_memory，检查数据解码和网络盘。",
        "forward": "检查 seq_len、batch、kernel 形状、混合精度。",
        "backward": "检查激活显存、activation checkpoint、DDP/FSDP 通信。",
        "optimizer": "检查优化器状态、fused optimizer、ZeRO/FSDP/distributed optimizer。",
    }
    return worst, ratios[worst], suggestions[worst]

examples = [
    {"dataloader": 40, "forward": 25, "backward": 30, "optimizer": 5},
    {"dataloader": 5, "forward": 20, "backward": 65, "optimizer": 10},
]
for row in examples:
    print(row, "=>", diagnose_step(row))


## 5. 与本课程的连接

- L01 的 `train_tiny_transformer.py` 会写 `dataloader_time_ms`、`forward_time_ms`、`backward_time_ms`、`optimizer_time_ms`。
- L02 后，backward 时间可能包含 DDP gradient all-reduce。
- L05 中，如果 recompute 打开，backward 变长不一定是坏事，它可能换来了更大 batch 或更长 seq。
- L07-L09 的 serving profiler 思路类似：把 e2e latency 拆成 queue、prefill、decode、network。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：为什么直接用 Python `time.time()` 测 GPU forward 可能不准？

**答案解析：** CUDA kernel launch 异步，CPU 可能只测到提交时间。需要 `torch.cuda.synchronize()` 或使用 profiler/CUDA event 才能测 GPU 实际完成时间。

### 题 2：GPU 利用率低一定是模型太小吗？

**答案解析：** 不一定。可能是 DataLoader 慢、CPU 同步太频繁、batch 太小、kernel launch overhead 高、通信等待、checkpoint I/O 或日志阻塞。Profiler 要先看 idle 区域对应的 CPU 侧事件。

### 题 3：`loss.item()` 为什么会影响性能？

**答案解析：** 它需要把 GPU tensor 的标量值同步回 CPU，会强制等待前面的 GPU 计算完成。偶尔记录没问题，每 step 大量 `.item()`/print 可能导致同步开销。

### 题 4：Profiler 中 backward 变长一定说明模型变慢了吗？

**答案解析：** 不一定。如果开启 activation checkpointing，backward 会重算 forward，时间变长是预期结果。需要同时看 peak memory、tokens/sec、是否允许更大 batch。

### 题 5：两个 trace 如何公平比较？

**答案解析：** 必须固定模型、batch、seq_len、dtype、数据路径、warmup、采样窗口和硬件；否则差异可能来自 workload，而不是代码改动。


## 7. 小结

Profiler 的核心不是“生成一张漂亮 trace”，而是把性能问题变成证据链：

```text
现象 → 分段指标 → trace 证据 → 最小假设 → 单变量实验 → 报告结论
```

下一步：运行 L01，打开 `artifacts/profiler/trace.json`，找出你机器上最大的 step time 组成。


## 参考资料

- PyTorch Profiler API: https://docs.pytorch.org/docs/stable/profiler.html
- PyTorch TensorBoard Profiler Tutorial: https://docs.pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html
- PyTorch CUDA semantics: https://docs.pytorch.org/docs/stable/notes/cuda.html
